# Beta-VAE arch x beta sweep

Joint sweep over `(base_channels, depth, beta)` at fixed `latent_dim=16`
(Stage 1, 60 configs), then a `(lr, weight_decay)` sweep at the Stage-1
winning config (Stage 2, 9 configs). Persists the final `(base, depth,
lr, weight_decay, beta)` winner's state_dict + metadata.

Sweep figures: Bousquet Fig 3 / Fig 5 replicas and a base x depth
heatmap of cell winners.


In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Anchor to the repo root so paleoreco imports and relative file paths
# resolve regardless of where Jupyter was launched from.
REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from paleoreco.data import (
    PaleoFieldDataset,
    apply_zscore,
    build_prior_cube,
    compute_zscore_stats,
)
from paleoreco.splits import block_stride_split, summarize_split
from paleoreco.models.autoencoder import (
    ConvAE,
    ConvBetaVAE,
    count_parameters,
)
from paleoreco.train_vae import train as train_vae
from paleoreco.train_ae import set_seed
from paleoreco.eval.ae import reconstruct_split
from paleoreco.eval.vae import (
    compute_vae_diagnostics,
    reconstruct_split_vae,
)
from paleoreco.eval.shared import compute_split_metrics

plt.rcParams["figure.dpi"] = 110


In [ ]:
SEED = 42
LATENT_DIM = 16
MAX_EPOCHS = 250
PATIENCE = 25
BATCH_SIZE = 32
KL_WARMUP_EPOCHS = 30
LOGVAR_CLAMP = (-10.0, 10.0)

# Stage 1: capacity x beta at v1.5-AE Stage-2 optimiser defaults.
BASE_CHANNELS = (8, 16, 32, 64)
DEPTHS = (2, 3, 4)
BETAS = (1e-5, 1e-4, 1e-3, 1e-2, 1e-1)
LR_STAGE1 = 3e-4
WD_STAGE1 = 1e-3

# Stage 2: optimiser grid at the Stage-1 winning (base, depth, beta).
LRS = (3e-4, 1e-3, 3e-3)
WDS = (1e-5, 1e-4, 1e-3)

# Selection: smallest val_mse_z subject to val_kl_total > KL_FLOOR
# (filters the not-learning regime where mu ~ 0 for every input).
KL_FLOOR = 1.0

OUT_DIR = Path(REPO_ROOT) / "outputs"
CSV_DIR = OUT_DIR / "csvs" / "05_vae_sweep"
FIG_DIR = OUT_DIR / "figures" / "05_vae_sweep"
CKPT_DIR = OUT_DIR / "checkpoints" / "05_vae_sweep"
STAGE1_CSV = CSV_DIR / "stage1.csv"
STAGE2_CSV = CSV_DIR / "stage2.csv"
WINNER_PT = CKPT_DIR / "vae_sweep_winner.pt"
AE_WINNER_PT = OUT_DIR / "checkpoints" / "03_ae_arch_sweep" / "ae_sweep_winner.pt"

for d in (OUT_DIR, CSV_DIR, FIG_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)


## Data and split


In [ ]:
data = build_prior_cube(
    prior_csv=os.path.join(REPO_ROOT, "data/Prior.csv"),
    cache_path=os.path.join(REPO_ROOT, "data/cache/prior_cube.npz"),
)
cube = data["cube"]
ages = data["ages"]
lats = data["lats"]
lons = data["lons"]
valid = data["valid"]
N_AGES = len(ages)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"cube: {cube.shape}  ages: [{ages.min()}, {ages.max()}] yr BP")
print(f"device: {device}")

split = block_stride_split(N_AGES)
print(summarize_split(ages, split))

stats = compute_zscore_stats(cube, train_age_indices=split["train"], valid=valid)
cube_z = apply_zscore(cube, stats)
mask = stats["safe_valid"]
print(f"\nsafe_valid cells: {int(mask.sum())} / {mask.size}")

train_ds = PaleoFieldDataset(cube_z, mask, split["train"])
val_ds   = PaleoFieldDataset(cube_z, mask, split["val"])
test_ds  = PaleoFieldDataset(cube_z, mask, split["test"])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
print(f"sizes: train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")


## AE reference

Re-evaluate the v1.5 AE sweep winner on this notebook's val split. The
split and stats are deterministic, so the AE's val metrics match its
original training run; here they serve as a horizontal reference on the
Bousquet Fig 3 replica below.


In [ ]:
ae_ckpt = torch.load(str(AE_WINNER_PT), map_location="cpu", weights_only=False)
ae_model = ConvAE(**ae_ckpt["config"])
ae_model.load_state_dict(ae_ckpt["state_dict"])
ae_model = ae_model.to(device)
ae_truth_z, ae_pred_z = reconstruct_split(ae_model, val_ds, device=device, batch_size=BATCH_SIZE)
ae_val = compute_split_metrics(ae_truth_z, ae_pred_z, mask)
AE_VAL_E_D = ae_val["E_d"]
AE_VAL_MSE_Z = ae_val["mse_z"]
print(f"AE reference (d={ae_ckpt['config']['latent_dim']}, base={ae_ckpt['config']['base_channels']}, depth={ae_ckpt['config']['depth']}):")
print(f"  val_mse_z={AE_VAL_MSE_Z:.4f}  val_E_d={AE_VAL_E_D:.4f}")


## Per-config training helper

Trains one VAE config end-to-end and returns its evaluation bundle. Val
metrics feed the CSV row and the selection criterion; test-side `<sigma>`
and `<D>` are kept in memory only, for the Bousquet Fig 5 replica.


In [ ]:
def train_one_config_vae(
    base_channels: int,
    depth: int,
    beta: float,
    lr: float,
    weight_decay: float,
) -> dict:
    """Train one VAE config; return state_dict, val metrics + diagnostics,
    test-side <sigma> and <D>, and meta.

    Sets the seed before model construction so init is deterministic;
    train_vae reseeds before its own DataLoader iteration so shuffling is
    the same across configs.
    """
    set_seed(SEED)
    model = ConvBetaVAE(
        latent_dim=LATENT_DIM,
        base_channels=base_channels,
        depth=depth,
        logvar_clamp=LOGVAR_CLAMP,
    )
    n_params = count_parameters(model)

    t0 = time.perf_counter()
    out = train_vae(
        model, train_loader, val_loader,
        mask=mask, zscore_std=stats["std"],
        beta=beta, kl_warmup_epochs=KL_WARMUP_EPOCHS,
        lr=lr, weight_decay=weight_decay,
        max_epochs=MAX_EPOCHS, patience=PATIENCE,
        device=device, seed=SEED,
        verbose=False, progress=True,
    )
    seconds = time.perf_counter() - t0

    # Re-evaluate val + test under best-val weights for the full metric +
    # diagnostic bundle (train_vae's history tracks the per-epoch scalars
    # but not per-dim KL or mu_norm).
    model.load_state_dict(out["best_state_dict"])
    val_truth, val_pred, val_mu, val_lv = reconstruct_split_vae(
        model, val_ds, device=device, batch_size=BATCH_SIZE,
    )
    val_metrics = compute_split_metrics(val_truth, val_pred, mask)
    val_diag = compute_vae_diagnostics(val_mu, val_lv)

    test_truth, test_pred, test_mu, test_lv = reconstruct_split_vae(
        model, test_ds, device=device, batch_size=BATCH_SIZE,
    )
    # <sigma> = mean over (test, dims) of exp(0.5 * logvar). Bousquet Fig 5 left.
    test_sigma_mean = float(np.exp(0.5 * test_lv).mean())
    # <D> = mean pairwise L2 distance between mu vectors on test. Bousquet Fig 5 right.
    N_test = test_mu.shape[0]
    diffs = test_mu[:, None, :] - test_mu[None, :, :]
    pair_dists = np.sqrt((diffs ** 2).sum(axis=-1))
    iu = np.triu_indices(N_test, k=1)
    test_pair_dist_mean = float(pair_dists[iu].mean()) if iu[0].size else float("nan")

    return {
        "state_dict": out["best_state_dict"],
        "val_metrics": val_metrics,
        "val_diag": val_diag,
        "val_total_loss": out["best_val_loss"],
        "test_sigma_mean": test_sigma_mean,
        "test_pair_dist_mean": test_pair_dist_mean,
        "n_params": n_params,
        "epochs_trained": out["epochs_trained"],
        "best_epoch": out["best_epoch"],
        "seconds": seconds,
    }


## Stage 1: capacity x beta sweep (60 configs)

All at `lr=LR_STAGE1`, `weight_decay=WD_STAGE1`. Joint over `(base, depth,
beta)` so the arch x beta interaction Bousquet flags (his Fig 3c) is
visible. CSV row written incrementally so a kernel crash on run k
preserves rows 1..k-1.


In [ ]:
stage1_rows: list[dict] = []
stage1_test_aux: list[dict] = []  # in-memory only; feeds Fig 5
stage1_configs = [
    (b, d, beta)
    for b in BASE_CHANNELS
    for d in DEPTHS
    for beta in BETAS
]

for i, (base, depth, beta) in enumerate(stage1_configs, start=1):
    print(f"\n[Stage 1 {i:>2}/{len(stage1_configs)}] base={base}  depth={depth}  beta={beta:.0e}")
    result = train_one_config_vae(
        base_channels=base, depth=depth, beta=beta,
        lr=LR_STAGE1, weight_decay=WD_STAGE1,
    )
    m = result["val_metrics"]
    diag = result["val_diag"]
    row = {
        "base_channels":          base,
        "depth":                  depth,
        "beta":                   beta,
        "val_mse_z":              m["mse_z"],
        "val_rmse_z":             m["rmse_z"],
        "val_rrmse":              m["rrmse"],
        "val_r_squared":          m["r_squared"],
        "val_E_d":                m["E_d"],
        "val_kl_total":           diag["kl_total"],
        "val_kl_per_dim_max":     float(diag["kl_per_dim"].max()),
        "val_kl_per_dim_min":     float(diag["kl_per_dim"].min()),
        "val_total_loss":         result["val_total_loss"],
        "val_mu_norm":            diag["mu_norm"],
        "val_post_cov_diag_mean": diag["post_cov_diag_mean"],
        "n_params":               result["n_params"],
        "epochs_trained":         result["epochs_trained"],
        "best_epoch":             result["best_epoch"],
    }
    stage1_rows.append(row)
    stage1_test_aux.append({
        "base_channels":       base,
        "depth":               depth,
        "beta":                beta,
        "test_sigma_mean":     result["test_sigma_mean"],
        "test_pair_dist_mean": result["test_pair_dist_mean"],
    })
    pd.DataFrame(stage1_rows).to_csv(STAGE1_CSV, index=False)

    print(
        f"  val_mse_z={m['mse_z']:.4f}  E_d={m['E_d']:.3f}  "
        f"KL_total={diag['kl_total']:.3f}  mu_norm={diag['mu_norm']:.3f}  "
        f"epochs={result['epochs_trained']}  ({result['seconds']:.0f}s)"
    )

stage1_df = pd.read_csv(STAGE1_CSV)
stage1_aux_df = pd.DataFrame(stage1_test_aux)
print(f"\nStage 1 complete: {len(stage1_df)} rows written to {STAGE1_CSV.relative_to(OUT_DIR.parent)}")


## Stage 1 figures


In [ ]:
# Bousquet Fig 3 replica: E_d (val) vs beta (left), aggregate KL vs beta
# (right), one line per (base, depth). Colour by base, linestyle by depth.
BASE_COLORS = {b: plt.get_cmap("viridis")(i / max(len(BASE_CHANNELS) - 1, 1))
               for i, b in enumerate(BASE_CHANNELS)}
DEPTH_STYLES = {2: "-", 3: "--", 4: ":"}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)

ax = axes[0]
for base in BASE_CHANNELS:
    for depth in DEPTHS:
        sub = stage1_df[(stage1_df["base_channels"] == base) & (stage1_df["depth"] == depth)]
        sub = sub.sort_values("beta")
        ax.plot(
            sub["beta"], sub["val_E_d"],
            color=BASE_COLORS[base], ls=DEPTH_STYLES[depth], marker="o", lw=1.4,
            label=f"base={base}, depth={depth}",
        )
ax.axhline(AE_VAL_E_D, color="k", lw=1.0, alpha=0.7, label=f"AE winner ({AE_VAL_E_D:.3f})")
ax.set_xscale("log")
ax.set_xlabel(r"$\beta$")
ax.set_ylabel(r"$E_d$ (val)")
ax.set_title("Bousquet Fig 3 replica: val $E_d$ vs beta")
ax.grid(True, alpha=0.3, which="both")
ax.legend(fontsize=6, ncol=2, loc="lower left")

ax = axes[1]
for base in BASE_CHANNELS:
    for depth in DEPTHS:
        sub = stage1_df[(stage1_df["base_channels"] == base) & (stage1_df["depth"] == depth)]
        sub = sub.sort_values("beta")
        ax.plot(
            sub["beta"], sub["val_kl_total"],
            color=BASE_COLORS[base], ls=DEPTH_STYLES[depth], marker="o", lw=1.4,
        )
ax.axhline(KL_FLOOR, color="r", lw=0.8, alpha=0.7, label=f"KL floor ({KL_FLOOR})")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$\beta$")
ax.set_ylabel(r"aggregate KL (val, nats)")
ax.set_title("Aggregate KL vs beta")
ax.grid(True, alpha=0.3, which="both")
ax.legend(fontsize=8, loc="lower right")

fig.savefig(FIG_DIR / "stage1_bousquet_fig3.png", bbox_inches="tight", dpi=120)
plt.show()


In [ ]:
# Bousquet Fig 5 replica: <sigma> = mean of exp(0.5 * logvar) on test;
# <D> = mean pairwise L2 distance between mu vectors on test. Same
# colour/style scheme as Fig 3.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)

ax = axes[0]
for base in BASE_CHANNELS:
    for depth in DEPTHS:
        sub = stage1_aux_df[(stage1_aux_df["base_channels"] == base) & (stage1_aux_df["depth"] == depth)]
        sub = sub.sort_values("beta")
        ax.plot(
            sub["beta"], sub["test_sigma_mean"],
            color=BASE_COLORS[base], ls=DEPTH_STYLES[depth], marker="o", lw=1.4,
            label=f"base={base}, depth={depth}",
        )
ax.axhline(1.0, color="k", lw=0.8, alpha=0.5, ls="--", label="prior std (1.0)")
ax.set_xscale("log")
ax.set_xlabel(r"$\beta$")
ax.set_ylabel(r"$\langle\sigma\rangle$ (test)")
ax.set_title("Bousquet Fig 5 replica: posterior $\\langle\\sigma\\rangle$")
ax.grid(True, alpha=0.3, which="both")
ax.legend(fontsize=6, ncol=2, loc="lower right")

ax = axes[1]
for base in BASE_CHANNELS:
    for depth in DEPTHS:
        sub = stage1_aux_df[(stage1_aux_df["base_channels"] == base) & (stage1_aux_df["depth"] == depth)]
        sub = sub.sort_values("beta")
        ax.plot(
            sub["beta"], sub["test_pair_dist_mean"],
            color=BASE_COLORS[base], ls=DEPTH_STYLES[depth], marker="^", lw=1.4,
        )
ax.set_xscale("log")
ax.set_xlabel(r"$\beta$")
ax.set_ylabel(r"$\langle D\rangle$ (test)")
ax.set_title("Bousquet Fig 5 replica: mean pairwise distance between $\\mu$")
ax.grid(True, alpha=0.3, which="both")

fig.savefig(FIG_DIR / "stage1_bousquet_fig5.png", bbox_inches="tight", dpi=120)
plt.show()


In [ ]:
# Stage-1 heatmap: per (base, depth) cell, best beta after KL-floor filter
# (annotated) coloured by the cell winner's val_mse_z.
cell_winners = []
for base in BASE_CHANNELS:
    for depth in DEPTHS:
        sub = stage1_df[(stage1_df["base_channels"] == base) & (stage1_df["depth"] == depth)]
        eligible = sub[sub["val_kl_total"] > KL_FLOOR]
        chosen = eligible if not eligible.empty else sub  # fall back if every beta collapses
        winner = chosen.sort_values("val_mse_z").iloc[0]
        cell_winners.append({
            "base_channels": base,
            "depth":         depth,
            "beta":          float(winner["beta"]),
            "val_mse_z":     float(winner["val_mse_z"]),
            "all_collapsed": bool(eligible.empty),
        })
cell_winners_df = pd.DataFrame(cell_winners)

grid_mse = cell_winners_df.pivot(index="depth", columns="base_channels", values="val_mse_z")
grid_beta = cell_winners_df.pivot(index="depth", columns="base_channels", values="beta")

fig, ax = plt.subplots(figsize=(6.5, 4.5), constrained_layout=True)
im = ax.imshow(grid_mse.values, aspect="auto", cmap="viridis_r")
ax.set_xticks(range(len(grid_mse.columns)), [f"{b}" for b in grid_mse.columns])
ax.set_yticks(range(len(grid_mse.index)), [f"{d}" for d in grid_mse.index])
ax.set_xlabel("base_channels")
ax.set_ylabel("depth")
ax.set_title("Stage-1 cell winners: val_mse_z (annot = best beta)")
for i in range(grid_mse.shape[0]):
    for j in range(grid_mse.shape[1]):
        ax.text(
            j, i, f"{grid_mse.values[i, j]:.3f}\nbeta={grid_beta.values[i, j]:.0e}",
            ha="center", va="center", color="w", fontsize=8,
        )
fig.colorbar(im, ax=ax, label="val_mse_z")
fig.savefig(FIG_DIR / "stage1_heatmap.png", bbox_inches="tight", dpi=120)
plt.show()

n_collapsed_cells = int(cell_winners_df["all_collapsed"].sum())
if n_collapsed_cells:
    print(f"WARNING: {n_collapsed_cells} (base, depth) cells had every beta below KL_FLOOR; their cell winner ignores the floor.")


## Stage 1 winner pick

Smallest `val_mse_z` subject to `val_kl_total > KL_FLOOR`. Tie-breaker
among near-equal configs is by-eye in favour of smaller `n_params`, not a
coded rule; print the top entries so the call is visible.


In [ ]:
stage1_eligible = stage1_df[stage1_df["val_kl_total"] > KL_FLOOR]
if stage1_eligible.empty:
    raise RuntimeError(
        f"No Stage-1 config satisfies val_kl_total > {KL_FLOOR}; the entire "
        "sweep landed in the not-learning regime. Lower KL_FLOOR or widen BETAS."
    )
stage1_sorted = stage1_eligible.sort_values("val_mse_z").reset_index(drop=True)
print("Top 5 Stage-1 candidates (KL_floor-filtered):")
print(stage1_sorted[[
    "base_channels", "depth", "beta",
    "val_mse_z", "val_E_d", "val_kl_total",
    "n_params", "epochs_trained",
]].head().to_string(index=False))

winner1 = stage1_sorted.iloc[0]
BASE_STAR = int(winner1["base_channels"])
DEPTH_STAR = int(winner1["depth"])
BETA_STAR = float(winner1["beta"])
print(
    f"\nStage 1 winner: base={BASE_STAR}, depth={DEPTH_STAR}, beta={BETA_STAR:.0e}  "
    f"(val_mse_z={float(winner1['val_mse_z']):.4f}, val_E_d={float(winner1['val_E_d']):.3f}, val_kl_total={float(winner1['val_kl_total']):.3f})"
)


## Stage 2: lr x weight_decay sweep at the Stage-1 winner (9 configs)

Beta is held at `BETA_STAR` so KL behaviour is pinned; the KL floor is
not re-applied. Selection: smallest `val_mse_z`.


In [ ]:
stage2_rows: list[dict] = []
stage2_states: dict[tuple[float, float], dict[str, torch.Tensor]] = {}
stage2_meta: dict[tuple[float, float], dict] = {}
stage2_configs = [(lr, wd) for lr in LRS for wd in WDS]

for i, (lr, wd) in enumerate(stage2_configs, start=1):
    print(f"\n[Stage 2 {i}/{len(stage2_configs)}] lr={lr:.0e}  wd={wd:.0e}")
    result = train_one_config_vae(
        base_channels=BASE_STAR, depth=DEPTH_STAR, beta=BETA_STAR,
        lr=lr, weight_decay=wd,
    )
    m = result["val_metrics"]
    diag = result["val_diag"]
    row = {
        "lr":                     lr,
        "weight_decay":           wd,
        "val_mse_z":              m["mse_z"],
        "val_rmse_z":             m["rmse_z"],
        "val_rrmse":              m["rrmse"],
        "val_r_squared":          m["r_squared"],
        "val_E_d":                m["E_d"],
        "val_kl_total":           diag["kl_total"],
        "val_kl_per_dim_max":     float(diag["kl_per_dim"].max()),
        "val_kl_per_dim_min":     float(diag["kl_per_dim"].min()),
        "val_total_loss":         result["val_total_loss"],
        "val_mu_norm":            diag["mu_norm"],
        "val_post_cov_diag_mean": diag["post_cov_diag_mean"],
        "n_params":               result["n_params"],
        "epochs_trained":         result["epochs_trained"],
        "best_epoch":             result["best_epoch"],
    }
    stage2_rows.append(row)
    stage2_states[(lr, wd)] = result["state_dict"]
    stage2_meta[(lr, wd)] = {
        "val_total_loss": result["val_total_loss"],
        "val_mse_z":      m["mse_z"],
        "val_kl_total":   diag["kl_total"],
        "best_epoch":     result["best_epoch"],
    }
    pd.DataFrame(stage2_rows).to_csv(STAGE2_CSV, index=False)

    print(
        f"  val_mse_z={m['mse_z']:.4f}  E_d={m['E_d']:.3f}  "
        f"KL_total={diag['kl_total']:.3f}  "
        f"epochs={result['epochs_trained']}  ({result['seconds']:.0f}s)"
    )

stage2_df = pd.read_csv(STAGE2_CSV)
print(f"\nStage 2 complete: {len(stage2_df)} rows written to {STAGE2_CSV.relative_to(OUT_DIR.parent)}")


## Stage 2 figure


In [ ]:
grid = stage2_df.pivot(index="lr", columns="weight_decay", values="val_mse_z")
fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
im = ax.imshow(grid.values, aspect="auto", cmap="viridis_r")
ax.set_xticks(range(len(grid.columns)), [f"{w:.0e}" for w in grid.columns])
ax.set_yticks(range(len(grid.index)),   [f"{l:.0e}" for l in grid.index])
ax.set_xlabel("weight_decay")
ax.set_ylabel("lr")
ax.set_title(f"Stage 2 val_mse_z at base={BASE_STAR}, depth={DEPTH_STAR}, beta={BETA_STAR:.0e}")
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        ax.text(j, i, f"{grid.values[i, j]:.4f}", ha="center", va="center",
                color="w", fontsize=9)
fig.colorbar(im, ax=ax, label="val_mse_z")
fig.savefig(FIG_DIR / "stage2_lr_wd_grid.png", bbox_inches="tight", dpi=120)
plt.show()


## Stage 2 winner pick


In [ ]:
stage2_sorted = stage2_df.sort_values("val_mse_z").reset_index(drop=True)
print("Stage 2 sorted by val_mse_z:")
print(stage2_sorted[[
    "lr", "weight_decay",
    "val_mse_z", "val_E_d", "val_kl_total",
    "epochs_trained",
]].to_string(index=False))

winner2 = stage2_sorted.iloc[0]
LR_STAR = float(winner2["lr"])
WD_STAR = float(winner2["weight_decay"])
WINNER_STATE = stage2_states[(LR_STAR, WD_STAR)]
WINNER_META = stage2_meta[(LR_STAR, WD_STAR)]
print(
    f"\nStage 2 winner: lr={LR_STAR:.0e}, weight_decay={WD_STAR:.0e}  "
    f"(val_mse_z={float(winner2['val_mse_z']):.4f}, val_E_d={float(winner2['val_E_d']):.3f})"
)


## Persist winner

Writes `outputs/checkpoints/05_vae_sweep/vae_sweep_winner.pt` with the
trained state_dict, the full architecture config, the training metadata,
and the best-epoch val metrics. This is the only checkpoint produced by
the sweep.


In [ ]:
checkpoint = {
    "state_dict": WINNER_STATE,
    "config": {
        "latent_dim":    LATENT_DIM,
        "base_channels": BASE_STAR,
        "depth":         DEPTH_STAR,
        "in_channels":   3,
        "out_channels":  2,
        "grid_shape":    (32, 64),
    },
    "training": {
        "beta":             BETA_STAR,
        "lr":               LR_STAR,
        "weight_decay":     WD_STAR,
        "max_epochs":       MAX_EPOCHS,
        "patience":         PATIENCE,
        "kl_warmup_epochs": KL_WARMUP_EPOCHS,
        "logvar_clamp":     LOGVAR_CLAMP,
        "seed":             SEED,
        "split":            "block_stride_80_10_10",
        "monitor":          "val_total_loss",
    },
    "metrics_at_early_stop": {
        "val_total_loss": WINNER_META["val_total_loss"],
        "val_mse_z":      WINNER_META["val_mse_z"],
        "val_kl_total":   WINNER_META["val_kl_total"],
    },
    "best_epoch": WINNER_META["best_epoch"],
}
torch.save(checkpoint, WINNER_PT)
print(f"saved {WINNER_PT}")
print(f"  config:    {checkpoint['config']}")
print(f"  training:  {checkpoint['training']}")
print(f"  metrics:   {checkpoint['metrics_at_early_stop']}")
print(f"  best_epoch: {checkpoint['best_epoch']}")
